In [1]:
# Cell 1
# Implementation of a KAN for Brain Voxel Classification (1-vs-Rest)
# This notebook implements a Kolmogorov-Arnold Network (KAN) for 
# brain voxel classification using a 1-vs-rest approach.

# Cell 1: Initialize the environment and import libraries
import torch
from kan import *
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import pandas as pd
import random
import glob
from tqdm import tqdm

# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

# Create folders for saving results
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('video_img', exist_ok=True)



Using GPU: NVIDIA RTX A6000


In [2]:
# Cell 2: Hyperparameters Configuration

class Config:
    def __init__(self):
        # Data parameters
        self.feature_dim = 341  # Number of input features
        self.num_classes = 2    # Binary classification (1-vs-rest)
        self.negative_ratio = 10  # Ratio of negative to positive samples (1:k)
        self.val_negative_ratio = 5  # Validation set negative to positive ratio
        
        # KAN model parameters
        self.network_width = [self.feature_dim,1024, 64, self.num_classes]  # Network architecture
        self.grid_size = 20     # Fixed grid size (no grid expansion to avoid oscillation)
        self.k_value = 3        # Number of basis functions per dimension
        
        # Training parameters
        self.optimizer = "Adam"  # Optimizer type (Adam, SGD, etc.)
        self.learning_rate = 0.005  # Learning rate for optimizer
        self.lambda_reg = 0.01    # Weight regularization coefficient
        self.lambda_entropy = 5.0  # Entropy regularization coefficient
        self.batch_size = 128      # Batch size for training
        self.train_steps = 1000    # Number of training steps
        self.early_stopping = False  # Whether to use early stopping
        self.patience = 10        # Early stopping patience
        self.min_delta = 0.001    # Minimum improvement for early stopping

        # Class weighting for imbalanced data
        self.class_weights = torch.tensor([1.0, self.negative_ratio * 0.5], dtype=torch.float32)  # [pos_weight, neg_weight]
        
        # Pruning and fine-tuning
        self.enable_pruning = True  # Whether to prune the model after training
        self.fine_tune_steps = 50   # Number of fine-tuning steps after pruning
    
        
        # Visualization parameters
        self.save_figures = False  # Whether to save figures
        self.img_folder = 'video_img'  # Folder to save training visualization images
        self.video_fps = 10        # Frames per second for training video
        
        # Symbolic regression parameters
        self.enable_symbolic = True  # Whether to extract symbolic expressions
        self.symbolic_library = ['x', 'x^2', 'exp', 'log', 'sqrt', 'sin', 'tanh', 'abs']  # Function library

    def display(self):
        """Display the current configuration"""
        print("=== KAN Brain Classification Configuration ===")
        print(f"Network Structure: {self.network_width}")
        print(f"Grid Size: {self.grid_size}, K Value: {self.k_value}")
        print(f"Negative to Positive Ratio: {self.negative_ratio}:1")
        print(f"Training Steps: {self.train_steps}, Batch Size: {self.batch_size}")
        print(f"Regularization: λ_reg={self.lambda_reg}, λ_entropy={self.lambda_entropy}")
        print(f"Learning Rate: {self.learning_rate}")
        print(f"Early Stopping: {self.early_stopping} (patience={self.patience}, min_delta={self.min_delta})")
        print(f"Class Weights: {self.class_weights}")
        print("===============================================")

# Create a configuration object
config = Config()
config.display()

=== KAN Brain Classification Configuration ===
Network Structure: [341, 1024, 64, 2]
Grid Size: 20, K Value: 3
Negative to Positive Ratio: 10:1
Training Steps: 1000, Batch Size: 128
Regularization: λ_reg=0.01, λ_entropy=5.0
Learning Rate: 0.005
Early Stopping: False (patience=10, min_delta=0.001)
Class Weights: tensor([1., 5.])


In [3]:
# Cell 3 修改: 训练集只加载一个batch，验证集使用全部label15数据+100倍负样本
# 移除了固定随机种子，使用真正的随机采样
def load_brain_voxel_batch(label_id, batch_size=1000, negative_ratio=10, val_negative_ratio=100):
    """
    训练集只加载一个batch的脑体素数据，验证集加载全部正样本和100倍负样本
    使用真正的随机采样，不设置固定种子
    """
    # 路径定义
    train_label_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/train_set_by_label'
    val_label_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output/val_set_by_label'
    
    # 1. 加载部分训练正样本
    positive_file = os.path.join(train_label_dir, f"label_{label_id}_count_*_voxels.npy")
    positive_files = glob.glob(positive_file)
    
    if not positive_files:
        raise ValueError(f"找不到标签{label_id}的训练数据文件")
    
    # 加载全部正样本
    all_positive = np.load(positive_files[0])
    
    # 仅选择batch_size/(negative_ratio+1)个正样本
    positive_count = batch_size // (negative_ratio + 1)
    if len(all_positive) > positive_count:
        # 真正随机选择，不使用固定种子
        indices = np.random.choice(len(all_positive), positive_count, replace=False)
        positive_samples = all_positive[indices]
    else:
        positive_samples = all_positive
        positive_count = len(positive_samples)
    
    print(f"加载了 {len(positive_samples)} 个训练正样本")
    
    # 2. 加载部分训练负样本
    negative_count = positive_count * negative_ratio
    negative_samples = []
    collected = 0
    
    label_files = glob.glob(os.path.join(train_label_dir, "label_*_count_*_voxels.npy"))
    other_label_files = [f for f in label_files if f"label_{label_id}_count" not in f]
    # 真正随机打乱
    random.shuffle(other_label_files)
    
    for file in other_label_files:
        if collected >= negative_count:
            break
            
        all_samples = np.load(file)
        to_take = min(len(all_samples), negative_count - collected)
        
        if to_take < len(all_samples):
            # 真正随机选择
            indices = np.random.choice(len(all_samples), int(to_take), replace=False)
            samples = all_samples[indices]
        else:
            samples = all_samples
            
        negative_samples.append(samples)
        collected += len(samples)
        
        # 打印添加了多少样本
        file_name = os.path.basename(file)
        sample_label = int(file_name.split('_')[1])
        print(f"  添加了 {len(samples)} 个训练负样本来自标签 {sample_label}")
        
        # 只从少数几个类别收集负样本，提高效率
        if len(negative_samples) >= 5:
            break
    
    negative_samples = np.vstack(negative_samples)
    if len(negative_samples) > negative_count:
        # 真正随机选择
        indices = np.random.choice(len(negative_samples), negative_count, replace=False)
        negative_samples = negative_samples[indices]
    
    print(f"收集了 {len(negative_samples)} 个训练负样本")
    
    # 3. 处理训练数据
    train_data = np.vstack([positive_samples, negative_samples])
    train_labels = np.concatenate([np.ones(len(positive_samples)), np.zeros(len(negative_samples))])
    
    # 随机打乱训练数据
    indices = np.random.permutation(len(train_data))
    train_data = train_data[indices]
    train_labels = train_labels[indices].astype(np.int64)
    
    # 4. 加载全部验证正样本
    val_positive_file = os.path.join(val_label_dir, f"label_{label_id}_count_*_voxels.npy")
    val_positive_files = glob.glob(val_positive_file)
    
    if not val_positive_files:
        print(f"警告: 找不到标签{label_id}的验证数据文件，将使用训练数据的一部分作为测试集")
        # 从训练数据中分出一部分作为测试集
        split_point = int(0.8 * len(train_data))
        test_data = train_data[split_point:]
        test_labels = train_labels[split_point:]
        train_data = train_data[:split_point]
        train_labels = train_labels[:split_point]
    else:
        # 加载全部验证正样本
        val_positive_samples = np.load(val_positive_files[0])
        val_positive_count = len(val_positive_samples)
        
        print(f"加载了 {val_positive_count} 个验证正样本（全部）")
        
        # 5. 加载验证负样本 - 100倍于正样本数量
        val_negative_count = val_positive_count * val_negative_ratio
        print(f"计划收集 {val_negative_count} 个验证负样本（{val_negative_ratio}倍于正样本）")
        
        val_negative_samples = []
        val_collected = 0
        
        val_label_files = glob.glob(os.path.join(val_label_dir, "label_*_count_*_voxels.npy"))
        val_other_label_files = [f for f in val_label_files if f"label_{label_id}_count" not in f]
        # 真正随机打乱
        random.shuffle(val_other_label_files)
        
        # 从每个可用的验证文件中获取负样本
        for file in val_other_label_files:
            if val_collected >= val_negative_count:
                break
                
            all_samples = np.load(file)
            to_take = min(len(all_samples), val_negative_count - val_collected)
            
            if to_take < len(all_samples):
                # 真正随机选择
                indices = np.random.choice(len(all_samples), int(to_take), replace=False)
                samples = all_samples[indices]
            else:
                samples = all_samples
                
            val_negative_samples.append(samples)
            val_collected += len(samples)
            
            # 打印每个标签添加的样本数
            file_name = os.path.basename(file)
            sample_label = int(file_name.split('_')[1])
            print(f"  添加了 {len(samples)} 个验证负样本来自标签 {sample_label}")
        
        val_negative_samples = np.vstack(val_negative_samples)
        if len(val_negative_samples) > val_negative_count:
            # 真正随机选择
            indices = np.random.choice(len(val_negative_samples), val_negative_count, replace=False)
            val_negative_samples = val_negative_samples[indices]
        
        print(f"实际收集了 {len(val_negative_samples)} 个验证负样本")
        
        # 创建测试数据
        test_data = np.vstack([val_positive_samples, val_negative_samples])
        test_labels = np.concatenate([np.ones(len(val_positive_samples)), np.zeros(len(val_negative_samples))])
        
        # 随机打乱测试数据
        indices = np.random.permutation(len(test_data))
        test_data = test_data[indices]
        test_labels = test_labels[indices].astype(np.int64)
    
    # 6. 转换为PyTorch张量
    train_inputs = torch.tensor(train_data, dtype=torch.float32).to(device)
    train_labels = torch.tensor(train_labels, dtype=torch.long).to(device)
    test_inputs = torch.tensor(test_data, dtype=torch.float32).to(device)
    test_labels = torch.tensor(test_labels, dtype=torch.long).to(device)
    
    # 创建类似Iris示例的数据集字典
    dataset = {
        'train_input': train_inputs,
        'train_label': train_labels,
        'test_input': test_inputs,
        'test_label': test_labels
    }
    
    # 检查训练集和测试集是否有重叠
    print("\n检查训练集和测试集是否有样本重叠...")
    # 为了节省计算资源，我们只检查一部分样本
    max_check = min(len(test_inputs), 100)  # 最多检查100个测试样本
    train_np = train_inputs.cpu().numpy()
    test_np = test_inputs[:max_check].cpu().numpy()
    
    overlaps = 0
    for i, test_sample in enumerate(test_np):
        # 计算与每个训练样本的欧氏距离
        distances = np.sqrt(np.sum((train_np - test_sample)**2, axis=1))
        min_dist = np.min(distances)
        if min_dist < 1e-6:  # 距离非常小，可能是相同样本
            overlaps += 1
            if overlaps <= 5:  # 只打印前5个重叠样本的详细信息
                print(f"  测试样本 {i} 与训练集样本重叠 (最小距离: {min_dist})")
    
    overlap_ratio = overlaps / max_check if max_check > 0 else 0
    print(f"  测试集中检查的 {max_check} 个样本中有 {overlaps} 个与训练集重叠 ({overlap_ratio:.2%})")
    
    print(f"\n训练集: {len(train_inputs)}个样本, 正样本={torch.sum(train_labels==1).item()}个, 负样本={torch.sum(train_labels==0).item()}个")
    print(f"测试集: {len(test_inputs)}个样本, 正样本={torch.sum(test_labels==1).item()}个, 负样本={torch.sum(test_labels==0).item()}个")
    
    return dataset

# 加载一个batch的训练数据和全部验证数据
target_label = 15
batch_size = 1000  # 训练批次大小
dataset = load_brain_voxel_batch(
    target_label, 
    batch_size=batch_size,  # 这里仍然使用本地定义的batch_size
    negative_ratio=config.negative_ratio,  # 10
    val_negative_ratio=config.val_negative_ratio  # 应该是5，但函数默认值是100
)
# 打印数据集信息
print(f"Train data shape: {dataset['train_input'].shape}")
print(f"Train target shape: {dataset['train_label'].shape}")
print(f"Test data shape: {dataset['test_input'].shape}")
print(f"Test target shape: {dataset['test_label'].shape}")
print("====================================")

加载了 90 个训练正样本
  添加了 900 个训练负样本来自标签 86
收集了 900 个训练负样本
加载了 272 个验证正样本（全部）
计划收集 1360 个验证负样本（5倍于正样本）
  添加了 1088 个验证负样本来自标签 7
  添加了 272 个验证负样本来自标签 69
实际收集了 1360 个验证负样本

检查训练集和测试集是否有样本重叠...
  测试集中检查的 100 个样本中有 0 个与训练集重叠 (0.00%)

训练集: 990个样本, 正样本=90个, 负样本=900个
测试集: 1632个样本, 正样本=272个, 负样本=1360个
Train data shape: torch.Size([990, 341])
Train target shape: torch.Size([990])
Test data shape: torch.Size([1632, 341])
Test target shape: torch.Size([1632])


In [4]:
# Cell 4: KAN Model Initialization (修改版)

import torch
from kan import *
import matplotlib.pyplot as plt

# 使用配置对象中的参数而不是硬编码值
model = KAN(width=config.network_width, grid=config.grid_size, k=config.k_value, device=device)
# 注意: 这里使用了config.network_width=[341, 128, 64, 2]，而不是之前硬编码的[341, 64, 32, 2]

# 执行前向传播来初始化模型
_ = model(dataset['train_input'][:10])

print(f"KAN模型已创建: 输入维度={config.network_width[0]}, 输出维度={config.network_width[-1]}")
print(f"网格大小: {config.grid_size}, k值: {config.k_value}")
print(f"总参数数量: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# # 尝试绘制模型结构
# try:
#     model.plot(beta=100, scale=1, in_vars=['F1', 'F2', 'F3'], out_vars=['Neg', 'Pos'])
# except:
#     print("无法绘制模型结构图，但这不影响训练")

checkpoint directory created: ./model
saving model version 0.0
KAN模型已创建: 输入维度=[341, 0], 输出维度=[2, 0]
网格大小: 20, k值: 3
总参数数量: 12030592


In [5]:
# Cell 5: Enhanced Training with Comprehensive Metrics (修改版)

from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score, precision_score, recall_score

# 定义准确率度量函数 - 按照Iris示例
def train_acc():
    return torch.mean((torch.argmax(model(dataset['train_input']), dim=1) == dataset['train_label']).float())

def test_acc():
    return torch.mean((torch.argmax(model(dataset['test_input']), dim=1) == dataset['test_label']).float())

# 创建加权损失函数
pos_count = torch.sum(dataset['train_label'] == 1).item()
neg_count = torch.sum(dataset['train_label'] == 0).item()
print(f"训练集: 正样本={pos_count}个, 负样本={neg_count}个, 比例=1:{neg_count/pos_count:.1f}")

# 使用配置中的类别权重，或者根据实际数据比例重新计算
# 注意: 这里使用实际数据比例而不是config.class_weights，因为数据加载后的比例可能与预设不同

# 类别权重调整 - 少数类应该有更高的权重
pos_weight = neg_count / pos_count  # 正样本权重 = 负样本数/正样本数
neg_weight = 1.0  # 多数类保持权重为1
class_weights = torch.tensor([neg_weight, pos_weight], device=device)
print(f"类别权重: {class_weights}")
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

# 训练模型，使用配置中的参数
print("开始训练模型...")
# 注意: 使用config中的参数替代硬编码值
results = model.fit(dataset, 
                    opt=config.optimizer,  # 使用"Adam"
                    metrics=(train_acc, test_acc),
                    loss_fn=loss_fn, 
                    steps=config.train_steps,  # 使用100
                    lamb=config.lambda_reg,  # 使用0.01而不是硬编码的0.001
                    lamb_entropy=config.lambda_entropy,  # 使用5.0而不是硬编码的1.0
                    save_fig=config.save_figures,  # 使用False
                    img_folder=config.img_folder)  # 使用'video_img'

# 注意: 如果要启用早停，需添加以下参数:
# early_stopping=config.early_stopping,
# early_stopping_patience=config.patience,
# early_stopping_min_delta=config.min_delta,
# KAN库可能不直接支持早停，可能需要自定义实现

print("训练完成!")
print(f"最终训练准确率: {results['train_acc'][-1]:.4f}")
print(f"最终测试准确率: {results['test_acc'][-1]:.4f}")

# 计算详细的评估指标
def compute_detailed_metrics(model, data_input, data_label):
    """计算详细的性能指标"""
    model.eval()
    with torch.no_grad():
        # 获取模型预测
        logits = model(data_input)
        probs = torch.softmax(logits, dim=1)
        # 获取正类的概率 (第二列)
        pos_probs = probs[:, 1].cpu().numpy()
        # 获取预测类别
        predicted = torch.argmax(logits, dim=1).cpu().numpy()
        # 真实标签
        true_labels = data_label.cpu().numpy()
        
        # 计算基本指标
        accuracy = (predicted == true_labels).mean()
        precision = precision_score(true_labels, predicted, zero_division=0)
        recall = recall_score(true_labels, predicted, zero_division=0)
        f1 = f1_score(true_labels, predicted, zero_division=0)
        
        # 计算AUC-PR
        precision_curve, recall_curve, _ = precision_recall_curve(true_labels, pos_probs)
        auc_pr = average_precision_score(true_labels, pos_probs)
        
        # 计算Macro-F1和Weighted-F1
        macro_f1 = f1_score(true_labels, predicted, average='macro', zero_division=0)
        weighted_f1 = f1_score(true_labels, predicted, average='weighted', zero_division=0)
        
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc_pr': auc_pr,
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1
        }

# 计算训练集和测试集的详细指标
train_metrics = compute_detailed_metrics(model, dataset['train_input'], dataset['train_label'])
test_metrics = compute_detailed_metrics(model, dataset['test_input'], dataset['test_label'])

# 打印详细评估结果
print("\n详细性能评估:")
print("                     训练集         测试集")
print(f"准确率:          {train_metrics['accuracy']:.4f}       {test_metrics['accuracy']:.4f}")
print(f"F1分数(正样本):   {train_metrics['f1']:.4f}       {test_metrics['f1']:.4f}")
print(f"精确度(正样本):   {train_metrics['precision']:.4f}       {test_metrics['precision']:.4f}")
print(f"召回率(正样本):   {train_metrics['recall']:.4f}       {test_metrics['recall']:.4f}")
print(f"AUC-PR:          {train_metrics['auc_pr']:.4f}       {test_metrics['auc_pr']:.4f}")
print(f"Macro-F1:        {train_metrics['macro_f1']:.4f}       {test_metrics['macro_f1']:.4f}")
print(f"Weighted-F1:     {train_metrics['weighted_f1']:.4f}       {test_metrics['weighted_f1']:.4f}")

# 同样修改剪枝后的评估代码
def evaluate_after_pruning():
    # 这部分代码将在剪枝和微调后添加
    print("\n剪枝和微调后的详细性能评估:")
    train_metrics = compute_detailed_metrics(model, dataset['train_input'], dataset['train_label'])
    test_metrics = compute_detailed_metrics(model, dataset['test_input'], dataset['test_label'])
    
    print("                     训练集         测试集")
    print(f"准确率:          {train_metrics['accuracy']:.4f}       {test_metrics['accuracy']:.4f}")
    print(f"F1分数(正样本):   {train_metrics['f1']:.4f}       {test_metrics['f1']:.4f}")
    print(f"精确度(正样本):   {train_metrics['precision']:.4f}       {test_metrics['precision']:.4f}")
    print(f"召回率(正样本):   {train_metrics['recall']:.4f}       {test_metrics['recall']:.4f}")
    print(f"AUC-PR:          {train_metrics['auc_pr']:.4f}       {test_metrics['auc_pr']:.4f}")
    print(f"Macro-F1:        {train_metrics['macro_f1']:.4f}       {test_metrics['macro_f1']:.4f}")
    print(f"Weighted-F1:     {train_metrics['weighted_f1']:.4f}       {test_metrics['weighted_f1']:.4f}")

训练集: 正样本=90个, 负样本=900个, 比例=1:10.0
类别权重: tensor([ 1., 10.], device='cuda:0')
开始训练模型...


description:   0%|                                                         | 0/1000 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.29 GiB. GPU 0 has a total capacity of 47.53 GiB of which 378.19 MiB is free. Process 1856816 has 14.88 GiB memory in use. Process 1489785 has 2.66 GiB memory in use. Process 1189735 has 10.07 GiB memory in use. Process 2300719 has 636.00 MiB memory in use. Process 982906 has 16.73 GiB memory in use. Process 1045532 has 2.18 GiB memory in use. Of the allocated memory 1.84 GiB is allocated by PyTorch, and 34.38 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Cell 6: Model Pruning and Symbolic Expression (修改版)

# 只有在配置中启用剪枝时才执行
if config.enable_pruning:
    print("对模型进行剪枝...")
    model = model.prune()
    print("剪枝完成")

    # 微调剪枝后的模型
    print("微调剪枝后的模型...")
    # 使用配置中的微调步数
    results_1 = model.fit(dataset, 
                         opt=config.optimizer,
                         metrics=(train_acc, test_acc),
                         loss_fn=loss_fn, 
                         steps=config.fine_tune_steps,  # 使用50
                         lamb=config.lambda_reg, 
                         lamb_entropy=config.lambda_entropy)

    print(f"微调后训练准确率: {results_1['train_acc'][-1]:.4f}")
    print(f"微调后测试准确率: {results_1['test_acc'][-1]:.4f}")

    # 尝试绘制剪枝后的模型
    try:
        model.plot(scale=1, in_vars=['F1', 'F2', 'F3'], out_vars=['Neg', 'Pos'])
    except:
        print("无法绘制剪枝后的模型结构图")
else:
    print("根据配置，跳过模型剪枝步骤")
    # 确保后续代码能够正常运行
    results_1 = results

# 只有在配置中启用符号表达式提取时才执行
if config.enable_symbolic:
    print("尝试提取符号表达式...")
    try:
        # 使用配置中的符号库
        lib = config.symbolic_library
        model.auto_symbolic(lib=lib)
        
        # 获取符号公式
        formula1, formula2 = model.symbolic_formula()[0]
        
        print("\n负类符号表达式:")
        print(formula1)
        
        print("\n正类符号表达式:")
        print(formula2)
        
        # 尝试简化公式
        try:
            from sympy import simplify
            print("\n简化后的正类表达式:")
            print(simplify(formula2))
        except:
            print("无法简化公式")
    except Exception as e:
        print(f"提取符号表达式失败: {e}")
else:
    print("根据配置，跳过符号表达式提取")

# 评估剪枝和微调后的性能
if config.enable_pruning:
    evaluate_after_pruning()

In [ ]:
# # Cell 7: Neural Network Comparison (修改版)

# # 定义一个标准神经网络进行比较
# class BrainNet(nn.Module):
#     def __init__(self, input_dim, hidden_dims, output_dim):
#         super(BrainNet, self).__init__()
#         # 使用配置中的网络宽度，而不是硬编码的值
#         self.fc1 = nn.Linear(input_dim, hidden_dims[0])
#         self.relu = nn.ReLU()
#         self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1])
#         self.fc3 = nn.Linear(hidden_dims[1], output_dim)

#     def forward(self, x):
#         x = self.fc1(x)
#         x = self.relu(x)
#         x = self.fc2(x)
#         x = self.relu(x)
#         x = self.fc3(x)
#         return x

# # 训练神经网络模型，使用配置中的参数
# def train_model(model, train_loader, criterion, optimizer, num_epochs=100):
#     model.train()
#     for epoch in range(num_epochs):
#         for inputs, labels in train_loader:
#             inputs, labels = inputs.to(device), labels.to(device)
#             optimizer.zero_grad()
#             outputs = model(inputs)
#             loss = criterion(outputs, labels)
#             loss.backward()
#             optimizer.step()
        
#         if (epoch+1) % 10 == 0:
#             # 每10个epoch输出一次损失
#             print(f'神经网络训练: Epoch {epoch+1}, Loss: {loss.item():.4f}')

# # 评估神经网络模型
# def test_model(model, test_loader):
#     model.eval()
#     correct = 0
#     total = 0
#     with torch.no_grad():
#         for inputs, labels in test_loader:
#             inputs, labels = inputs.to(device), labels.to(device)
#             outputs = model(inputs)
#             _, predicted = torch.max(outputs.data, 1)
#             total += labels.size(0)
#             correct += (predicted == labels).sum().item()
    
#     accuracy = 100 * correct / total
#     print(f'神经网络测试准确率: {accuracy:.2f}%')
#     return accuracy

# # 创建数据加载器，使用配置中的批次大小
# train_loader = torch.utils.data.DataLoader(
#     torch.utils.data.TensorDataset(dataset['train_input'], dataset['train_label']), 
#     batch_size=config.batch_size,  # 使用64而不是硬编码的32
#     shuffle=True
# )

# test_loader = torch.utils.data.DataLoader(
#     torch.utils.data.TensorDataset(dataset['test_input'], dataset['test_label']), 
#     batch_size=config.batch_size,  # 使用64而不是硬编码的32
#     shuffle=False
# )

# # 初始化神经网络，使用与KAN相同的网络结构
# nn_model = BrainNet(
#     input_dim=config.network_width[0],  # 341
#     hidden_dims=[config.network_width[1], config.network_width[2]],  # [128, 64]
#     output_dim=config.network_width[-1]  # 2
# ).to(device)

# # 使用与KAN训练相同的权重损失函数
# criterion = nn.CrossEntropyLoss(weight=class_weights)
# optimizer = torch.optim.Adam(nn_model.parameters(), lr=config.learning_rate)  # 使用0.005而不是0.01

# # 训练神经网络
# print("开始训练神经网络进行比较...")
# train_model(nn_model, train_loader, criterion, optimizer, num_epochs=config.train_steps)  # 使用100而不是默认值

# # 评估神经网络
# nn_accuracy = test_model(nn_model, test_loader)

# # 比较KAN和神经网络
# print("\n模型比较:")
# print(f"KAN测试准确率: {results_1['test_acc'][-1]*100:.2f}%")
# print(f"神经网络测试准确率: {nn_accuracy:.2f}%")
# print(f"差异: {(results_1['test_acc'][-1]*100 - nn_accuracy):.2f}%")

# # 打印KAN的优势
# print("\nKAN的优势:")
# print("1. 可解释性 - 能够提取数学公式解释决策")
# print("2. 模型剪枝 - 可以减少模型大小并保持性能")
# print("3. 固定网格大小避免训练震荡")

In [ ]:
# # Cell 8: Create Video from Training Images (修改版)

# import os
# import numpy as np

# try:
#     import moviepy.video.io.ImageSequenceClip
    
#     # 创建视频，使用配置中的参数
#     video_name = 'video'
#     fps = config.video_fps  # 使用10而不是硬编码值
    
#     # 获取图像文件路径
#     image_folder = config.img_folder  # 使用'video_img'而不是硬编码值
#     files = os.listdir(image_folder)
#     train_index = []
    
#     # 获取所有数字命名的jpg文件
#     for file in files:
#         if file[0].isdigit() and file.endswith('.jpg'):
#             train_index.append(int(file[:-4]))
    
#     # 按正确顺序排序索引
#     train_index = np.sort(train_index)
    
#     # 创建图像文件路径列表
#     image_files = [f'{image_folder}/{idx}.jpg' for idx in train_index]
    
#     if image_files:
#         # 创建视频并保存
#         clip = moviepy.video.io.ImageSequenceClip.ImageSequenceClip(image_files, fps=fps)
#         clip.write_videofile(f'{video_name}.mp4')